# X-Ray Red vs Blue

Explore white-box evasion and lightweight defenses on the bundled chest X-ray classifier. The standard EfficientNet model stays fixed. Red tunes FGSM/PGD parameters run by the evaluator; Blue compares bundled preprocessing and detector options. No model training is required. This classifier is for education only, not medical diagnosis.

## Round flow

1. Each student runs Red and submits their own successful Red flag in CTFd. The first success freezes the shared artifact used by Blue; later Red successes earn the flag without replacing it.
2. Blue opens this notebook and loads the active Red artifact automatically.
3. Blue compares defenses. The evaluator checks clean usable accuracy, attack resolution, detection, and false alarms.
4. Submit any returned flag to the matching Red or Blue challenge in CTFd.

This prototype uses one shared artifact at a time. The instructor assigns Red and Blue roles; CTFd does not enforce team membership. Each account must submit its own Red flag before Blue opens. The first passing Blue defense completes the shared round, but later students may still evaluate and earn their own flags. An administrator can reset the evaluator after both phases finish.


## 1. Connect and check the round


In [ ]:
from range_device import select_device, describe_device
device = select_device()
print("Compute device:", describe_device(device))

import os
import time
from pathlib import Path

import numpy as np
import requests
from PIL import Image, ImageDraw
from IPython.display import display, Markdown

XRAY_URL = os.getenv("XRAY_REDBLUE_URL", "http://xray-redblue:5000").rstrip("/")
ARENA_TOKEN = os.getenv("ARENA_TOKEN") or os.getenv("JUPYTER_TOKEN", "")

def xray_headers():
    return {"Authorization": f"Bearer {ARENA_TOKEN}"} if ARENA_TOKEN else {}

def get_json(path, timeout=15):
    response = requests.get(f"{XRAY_URL}{path}", headers=xray_headers(), timeout=timeout)
    response.raise_for_status()
    return response.json()

try:
    participant_identity = get_json("/api/identity")
    print(f"MLflow attribution: {participant_identity['ctfd_username']} "
          f"(CTFd ID {participant_identity['ctfd_user_id']}; "
          f"resolved via {participant_identity['identity_source']})")
except requests.RequestException:
    print("MLflow attribution could not be confirmed; evaluation still works.")

health = get_json("/api/health")
config = get_json("/api/config")
round_state = get_json("/api/round")
print("Evaluator:", health["status"], "on", health["device"])
print("Balanced sample:", health["samples_per_class"], "per class")
print("Round phase:", round_state["phase"])
print("Allowed attacks:", ", ".join(config["attacks"]))
print("Blue preprocessors:", ", ".join(config["preprocessors"]))
print("Blue detectors:", ", ".join(config["detectors"]))


## 2. Red challenge: find a successful evasion

Choose FGSM or PGD and adjust epsilon. The server runs the white-box attack on a fixed balanced subset. A result succeeds when the attack flips enough originally correct predictions while staying within the perturbation budget. Keep epsilon at or below `0.02`. The first success publishes the shared artifact; later successes still return a Red flag for the student who ran them. Edit the call below and run it when you are ready to submit an attack.


In [ ]:
def wait_for_job(job_id, timeout=900, interval=3):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        job = get_json(f"/api/jobs/{job_id}")
        if job["status"] == "completed":
            return job["result"]
        if job["status"] == "failed":
            raise RuntimeError(job.get("error", "Exercise job failed"))
        time.sleep(interval)
    raise TimeoutError("The exercise job is still running; rerun the polling cell with the same job ID.")

def run_red_attack(attack="FGSM", epsilon=0.01, step=0.0025, iterations=3):
    response = requests.post(
        f"{XRAY_URL}/api/red/attack",
        headers=xray_headers(),
        json={"attack": attack, "epsilon": epsilon, "step": step, "iterations": iterations},
        timeout=30,
    )
    response.raise_for_status()
    job_id = response.json()["job_id"]
    print("Red job:", job_id)
    result = wait_for_job(job_id)
    print({key: result[key] for key in ("attack", "epsilon", "total_samples", "clean_accuracy", "adversarial_accuracy", "attack_success_rate", "measured_linf", "success")})
    if result.get("flag"):
        display(Markdown(f"**Red challenge flag:** `{result['flag']}` — submit it to the Red CTFd challenge."))
    return result

# Red team: choose FGSM or PGD and tune the parameters. For example,
# red_result = run_red_attack(attack="FGSM", epsilon=0.005)
red_result = None


## 3. Blue challenge: inspect the handed-off Red artifact

After Red succeeds, any Blue workspace can load the round's frozen artifact. The NPZ contains the clean/attacked sample pair and labels used for evaluation. The scoring rubric and limits are published in the evaluator configuration so teams can understand what the metrics mean.


In [ ]:
def load_latest_red_artifact():
    info = get_json("/api/round/latest")
    response = requests.get(f"{XRAY_URL}{info['artifact_url']}", headers=xray_headers(), timeout=60)
    response.raise_for_status()
    path = Path("red_submission.npz")
    path.write_bytes(response.content)
    with np.load(path, allow_pickle=False) as data:
        clean = data["x_clean"].copy()
        adversarial = data["x_adv"].copy()
        labels = data["labels"].copy()
    print("Loaded Red run:", info["result"]["run_id"], "·", len(labels), "images")
    return clean, adversarial, labels, info["result"]

def show_pair(index=0):
    clean = np.clip(clean_images[index].transpose(1, 2, 0) * 255, 0, 255).astype("uint8")
    adv = np.clip(red_images[index].transpose(1, 2, 0) * 255, 0, 255).astype("uint8")
    pair = Image.new("RGB", (clean.shape[1] * 2, clean.shape[0] + 26), "white")
    pair.paste(Image.fromarray(clean), (0, 26))
    pair.paste(Image.fromarray(adv), (clean.shape[1], 26))
    draw = ImageDraw.Draw(pair)
    draw.text((8, 6), "Clean", fill="black")
    draw.text((clean.shape[1] + 8, 6), "Red adversarial", fill="black")
    display(pair)

round_state = get_json("/api/round")
if round_state["phase"] in {"blue", "complete"}:
    clean_images, red_images, labels, active_red_result = load_latest_red_artifact()
    show_pair(0)
else:
    print("Waiting for Red to succeed. Rerun this cell after the round enters the Blue phase.")


## 4. Blue challenge: test a lightweight defense

Try the supplied robust EfficientNet checkpoint or the standard checkpoint with a preprocessor such as a 3×3 median filter, Gaussian blur, or quantization. You can also use either a noisy-view consistency detector or a confidence-gated disagreement detector that compares the standard and robust checkpoint predictions. These detectors flag suspicious inputs; they do not remove perturbations or change model weights. No further training is required.

Compare several settings by editing the parameters below. With the default evaluator settings, a successful Blue defense must retain at least 65% clean usable accuracy, resolve at least 55% of adversarial examples and improve on Red by 5 percentage points, detect at least 5% of adversarial examples, and keep clean false alarms at or below 25%. An adversarial example is resolved if the model predicts its true label or the detector flags it for review; a flag on a clean example counts as a false alarm. The detection floor means a model-only switch with no detector cannot pass. Choose and run a configuration to submit your Blue defense.


In [ ]:
def evaluate_blue(model="standard", preprocess="none", detector="none", confidence=0.6, bits=5, samples=4, sigma=0.03, threshold=0.75):
    response = requests.post(
        f"{XRAY_URL}/api/blue/defend",
        headers=xray_headers(),
        json={"model": model, "preprocess": preprocess, "detector": detector, "confidence": confidence, "bits": bits, "samples": samples, "sigma": sigma, "threshold": threshold},
        timeout=30,
    )
    response.raise_for_status()
    job_id = response.json()["job_id"]
    result = wait_for_job(job_id)
    display(Markdown("**Blue defense result**"))
    display(result)
    if result.get("flag"):
        display(Markdown(f"**Blue challenge flag:** `{result['flag']}` — submit it to the Blue CTFd challenge."))
    return result

# Blue team: choose a model, preprocessor, detector, and detector threshold.
# Examples: evaluate_blue(model="standard", preprocess="median3")
#           evaluate_blue(model="standard", preprocess="none", detector="disagreement", confidence=0.6)
blue_result = None


## 5. Resetting the round

A successful Red submission closes Red and freezes its artifact for Blue. After the round is complete, an administrator resets it with the protected evaluator API; do not put the admin token in this notebook. Restarting the container preserves the active round and artifact.
